# 08 — Systems Profiler Capstone

**Master syllabus coverage:** V2 Part I capstone

## Why this matters

The capstone integrates hardware, movement, precision, parallel work, matmul, benchmarking, and budgeting into a reproducible report and a set of falsifiable performance hypotheses.

## Learning objectives

- Build a reusable synchronized benchmark kernel.
- Report vector, matmul, layout, allocation, and precision experiments.
- Separate measurements from bottleneck hypotheses.
- Specify the next profiler or scaling experiment needed to validate each hypothesis.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## Capstone contract

Build a compact systems profiler that records environment, shape, dtype, median latency,
throughput, and a bottleneck hypothesis for representative numerical workloads. The goal
is not a perfect profiler; it is a disciplined workflow that replaces “the GPU is slow”
with testable hypotheses about compute, movement, precision, allocation, or launch cost.


In [ ]:
import platform
import time
import numpy as np
import torch

def synchronize(device):
    if device.type == "cuda": torch.cuda.synchronize(device)
    elif device.type == "mps": torch.mps.synchronize()

device = (torch.device("cuda") if torch.cuda.is_available() else
          torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu"))

def median_ms(function, *, warmup=3, repeats=15, target=device):
    for _ in range(warmup): function()
    synchronize(target)
    samples = []
    for _ in range(repeats):
        synchronize(target); start = time.perf_counter_ns(); function(); synchronize(target)
        samples.append((time.perf_counter_ns() - start) / 1e6)
    return float(np.median(samples))

environment = {
    "machine": platform.machine(), "OS": platform.platform(),
    "python": platform.python_version(), "torch": torch.__version__, "device": str(device),
}
print(environment)


## 1. Run controlled workloads

Each report row names its shape and dtype, includes an approximate amount of work, and
records the measurement method. The bottleneck label is a hypothesis to test with a real
profiler—not a conclusion derived automatically from one ratio.


In [ ]:
rows = []

vector = torch.randn(1_000_000, device=device)
vector_ms = median_ms(lambda: vector + 1)
rows.append({"workload": "vector add", "shape": list(vector.shape), "dtype": str(vector.dtype),
             "median_ms": vector_ms, "hypothesis": "bandwidth or launch overhead"})

left = torch.randn(512, 512, device=device)
right = torch.randn(512, 512, device=device)
matmul_ms = median_ms(lambda: left @ right)
rows.append({"workload": "matmul", "shape": [list(left.shape), list(right.shape)],
             "dtype": str(left.dtype), "median_ms": matmul_ms, "hypothesis": "compute/tiling"})

for row in rows: print(row)


## 2. Compare allocation with reuse and contiguous with strided

These demonstrations isolate memory-management and layout hypotheses. They use CPU NumPy
so their array semantics are visible; repeat on the future C++/WebGPU runtime before using
the results for engine design.


In [ ]:
rng = np.random.default_rng(42)
a = rng.normal(size=(1024, 1024)).astype(np.float32)
b = rng.normal(size=(1024, 1024)).astype(np.float32)
out = np.empty_like(a)
allocate_ms = median_ms(lambda: a + b, target=torch.device("cpu"))
reuse_ms = median_ms(lambda: np.add(a, b, out=out), target=torch.device("cpu"))

contiguous = a
strided = a.T
contiguous_ms = median_ms(lambda: contiguous.sum(axis=1), target=torch.device("cpu"))
strided_ms = median_ms(lambda: strided.sum(axis=1), target=torch.device("cpu"))
print({"allocate_ms": allocate_ms, "reuse_ms": reuse_ms,
       "contiguous_reduce_ms": contiguous_ms, "strided_reduce_ms": strided_ms})
np.testing.assert_array_equal(out, a + b)


## 3. Compare storage precision and reconstruction

Performance without numerical quality is incomplete. Record bytes and error when testing
lower precision or quantization, and identify which operations still accumulate in higher
precision.


In [ ]:
signal = rng.normal(size=100_000).astype(np.float32)
precision_rows = []
for dtype in (np.float32, np.float16):
    stored = signal.astype(dtype)
    restored = stored.astype(np.float32)
    precision_rows.append({"dtype": str(dtype), "bytes": stored.nbytes,
                           "RMSE": float(np.sqrt(np.mean((signal - restored) ** 2)))})
print(precision_rows)
assert precision_rows[1]["bytes"] == precision_rows[0]["bytes"] // 2


## 4. Produce the report and hypotheses

A useful capstone report includes what was measured, what was held constant, correctness
checks, and the next experiment. Avoid assigning “compute-bound” or “memory-bound” from
intuition alone; a profiler, bandwidth counters, or scaling experiment should test it.


In [ ]:
report = {
    "environment": environment,
    "workloads": rows,
    "precision": precision_rows,
    "method": {"warmup": 3, "repeats": 15, "statistic": "median", "synchronized": True},
    "next_experiments": [
        "sweep tensor size to find overhead/throughput transition",
        "measure effective bandwidth for elementwise operations",
        "profile matmul shapes used by the planned decoder",
        "measure peak memory rather than estimating only live arrays",
    ],
}
print(report)
assert len(report["workloads"]) >= 2 and report["method"]["synchronized"]


## Failure modes to recognize

- **Hardware folklore:** a bottleneck label is assigned without a discriminating measurement.
- **No correctness evidence:** a fast variant may compute different values.
- **Environment omitted:** results cannot be reproduced or compared.
- **Microbenchmark overreach:** one synthetic workload dictates product architecture.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Add small and large CPU/accelerator comparisons including transfer time.
2. Add effective bandwidth and approximate FLOP/s fields with clearly stated formulas.
3. Write a one-page report classifying each workload as an evidence-backed or still-untested hypothesis.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can inspect a slow tensor program and propose separate tests for compute, memory, transfer, allocation, precision, and launch overhead.

**Next:** 09 — Tensors and numerical Python.
